In [1]:
# full_prototype_with_fixes.py
# Jupyter-ready prototype (no TensorFlow).
# - RAG (sentence-transformers + faiss) + TF-IDF keywords
# - Few-shot prompting and difficulty-aware generation (easy/medium/hard)
# - Semantic deduplication
# - Mixed mode default split + optional custom proportions/counts (robust parser)
# - Fixed difficulty rounding & per-batch request logic
#
# Requirements:
# pip install sentence-transformers faiss-cpu scikit-learn PyMuPDF docx2txt openai tqdm

# ---------------- Jupyter-safe startup ----------------
import sys
sys.argv = [sys.argv[0]]

# ---------------- Imports ----------------
import os
import json
import uuid
import re
import math
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Any, Tuple

import fitz  # PyMuPDF
import docx2txt

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer

from openai import OpenAI

from tqdm import tqdm


C:\Users\moham\.conda\envs\jupyter-ml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ---------------- User config (edit locally) ----------------
# Keep this Tesseract line commented for future use:
# import pytesseract
# pytesseract.pytesseract.tesseract_cmd = r"C:\\Program Files\\Tesseract-OCR\\tesseract.exe"

OPENAI_API_KEY = "API KEY"   # <<-- paste your key here
if not OPENAI_API_KEY or OPENAI_API_KEY.startswith("sk-REPLACE"):
    raise RuntimeError("Set OPENAI_API_KEY variable in the script (paste your key string).")

client = OpenAI(api_key=OPENAI_API_KEY)

# Files: set exact filenames or absolute paths to PDFs/DOCX
# Files (edit to exact filenames or absolute paths)
FILES: List[str] = [
    # Put your PDF filename(s) here; example for your working folder:
    r"Introduction to Machine Learning with Python ( PDFDrive.com )-min-81-82.pdf"
]


OUTPUT_DIR = Path("./generated_tests")
OUTPUT_DIR.mkdir(exist_ok=True)

# Chunking params
CHUNK_CHARS = 3200
CHUNK_OVERLAP = 300

# Embedding model
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"

# FAISS / RAG params
TOP_K_DEFAULT = 4
DEDUP_SIM_THRESHOLD = 0.82

# OpenAI generation params
MODEL = "gpt-4o"
TEMPERATURE = 0.0
MAX_TOKENS = 900

# Difficulty split ratios (easy, medium, hard)
DIFF_RATIOS = (0.5, 0.3, 0.2)  # sum == 1

# Globals for FAISS & chunk meta
FAISS_INDEX = None
CHUNK_VECTORS = None
CHUNK_META: List[Dict[str, Any]] = []

In [3]:
# ---------------- Utilities ----------------
def now_ts():
    return datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


In [4]:
# ---------------- Extraction ----------------
def extract_text_from_pdf(path: str) -> str:
    doc = fitz.open(path)
    pages_text = []
    for i in range(len(doc)):
        page = doc.load_page(i)
        text = page.get_text("text").strip()
        # OCR fallback intentionally commented for now
        pages_text.append(f"===PAGE {i+1}===\n{text}\n")
    return "\n".join(pages_text)

def extract_text_from_docx(path: str) -> str:
    return docx2txt.process(path)

def extract_text_from_file(path: str) -> str:
    ext = Path(path).suffix.lower()
    if ext == ".pdf":
        return extract_text_from_pdf(path)
    elif ext in (".docx", ".doc"):
        return extract_text_from_docx(path)
    else:
        raise ValueError(f"Unsupported file extension: {ext}")

In [5]:
# ---------------- Fast paragraph chunker ----------------
def _normalize_whitespace(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def chunk_text_fast(text: str, chunk_chars: int = CHUNK_CHARS, overlap: int = CHUNK_OVERLAP) -> List[str]:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
    if not paragraphs:
        s = _normalize_whitespace(text)
        return [s] if s else []
    chunks: List[str] = []
    cur_parts: List[str] = []
    cur_len = 0
    for p in paragraphs:
        p_len = len(p)
        if p_len > chunk_chars:
            if cur_parts:
                chunks.append(" ".join(cur_parts).strip())
                cur_parts = []
                cur_len = 0
            i = 0
            while i < p_len:
                part = p[i:i+chunk_chars].strip()
                if part:
                    chunks.append(part)
                i += (chunk_chars - overlap) if (chunk_chars - overlap) > 0 else chunk_chars
            continue
        if cur_len + p_len + (1 if cur_parts else 0) <= chunk_chars:
            cur_parts.append(p)
            cur_len += p_len + (1 if cur_parts else 0)
        else:
            if cur_parts:
                chunks.append(" ".join(cur_parts).strip())
            cur_parts = [p]
            cur_len = p_len
    if cur_parts:
        chunks.append(" ".join(cur_parts).strip())
    if overlap > 0 and len(chunks) > 1:
        new_chunks = [chunks[0]]
        for i in range(1, len(chunks)):
            prev = new_chunks[-1]
            take = prev[-overlap:] if len(prev) > overlap else prev
            merged = (take + " " + chunks[i]).strip()
            new_chunks.append(merged)
        chunks = new_chunks
    chunks = [c for c in chunks if len(c) > 50]
    return chunks


In [6]:
# ---------------- TF-IDF keywords per chunk ----------------
def compute_chunk_keywords(chunks_texts: List[str], top_k: int = 6) -> List[List[str]]:
    vectorizer = TfidfVectorizer(ngram_range=(1,2), stop_words="english", max_features=5000)
    X = vectorizer.fit_transform(chunks_texts)
    feature_names = np.array(vectorizer.get_feature_names_out())
    keywords_per_chunk = []
    for i in range(X.shape[0]):
        row = X[i].toarray().ravel()
        if row.sum() == 0:
            keywords_per_chunk.append([])
            continue
        top_idx = np.argsort(-row)[:top_k]
        keywords = [feature_names[j] for j in top_idx if row[j] > 0]
        keywords_per_chunk.append(keywords)
    return keywords_per_chunk

In [7]:
# ---------------- Embedding + FAISS ----------------
def build_embeddings_index(chunks: List[Dict[str, Any]], model_name: str = EMBED_MODEL_NAME):
    global FAISS_INDEX, CHUNK_VECTORS
    texts = [c["text"] for c in chunks]
    print("Embedding chunks (this may take a moment)...")
    embed_model = SentenceTransformer(model_name)
    vecs = embed_model.encode(texts, convert_to_numpy=True, show_progress_bar=True, normalize_embeddings=True)
    vecs = vecs.astype("float32")
    d = vecs.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(vecs)
    FAISS_INDEX = index
    CHUNK_VECTORS = vecs
    print("FAISS index built with", FAISS_INDEX.ntotal, "vectors.")
    return embed_model

def retrieve_top_k(query: str, embed_model: SentenceTransformer, top_k: int = TOP_K_DEFAULT) -> List[Dict[str, Any]]:
    if FAISS_INDEX is None:
        raise RuntimeError("FAISS index not built")
    qv = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    qv = qv.astype("float32")
    D, I = FAISS_INDEX.search(qv, top_k)
    idxs = I[0].tolist()
    results = []
    for idx in idxs:
        if idx < 0 or idx >= len(CHUNK_META):
            continue
        results.append(CHUNK_META[idx])
    return results


In [8]:
# ---------------- OpenAI JSON call helper ----------------
def call_openai_json(prompt: str) -> Dict[str, Any]:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a JSON-only responder. Return only JSON."},
            {"role": "user", "content": prompt}
        ],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS
    )
    try:
        content = resp["choices"][0]["message"]["content"]
    except Exception:
        content = resp.choices[0].message.content
    txt = content.strip()
    start_idx = txt.find("{")
    if start_idx == -1:
        raise ValueError("OpenAI returned no JSON object:\n" + txt[:800])
    brace = 0
    end_idx = None
    for i in range(start_idx, len(txt)):
        if txt[i] == "{":
            brace += 1
        elif txt[i] == "}":
            brace -= 1
            if brace == 0:
                end_idx = i + 1
                break
    json_str = txt[start_idx:end_idx] if end_idx else txt
    try:
        return json.loads(json_str)
    except Exception:
        fixed = re.sub(r",\s*}", "}", json_str)
        fixed = re.sub(r",\s*\]", "]", fixed)
        return json.loads(fixed)


In [9]:
# ---------------- Few-shot examples ----------------
FEWSHOT_EXAMPLE_MCQ = """
{"type":"mcq","stem":"What is the primary purpose of L1 regularization?","options":["Increase model capacity","Reduce overfitting by feature selection","Speed up training","Normalize inputs"],"answer":"Reduce overfitting by feature selection","difficulty":"easy","confidence":0.95}
"""

FEWSHOT_EXAMPLE_YESNO = """
{"type":"yesno","stem":"Does L2 regularization drive weights to zero?","answer":"No","difficulty":"easy","confidence":0.9}
"""

FEWSHOT_EXAMPLE_CODE = """
{"type":"code","stem":"Implement L2 penalty function","description":"Write a Python function that computes L2 penalty for a weight vector.","template":"def l2_penalty(w):\\n    # TODO","tests":"import numpy as np\\ndef test_l2():\\n    assert abs(l2_penalty(np.array([1.0,2.0])) - 5.0) < 1e-6","difficulty":"medium","confidence":0.9}
"""

FEWSHOT_EXAMPLE_DESC = """
{"type":"descriptive","stem":"Explain what L1 regularization does to model weights.","reference_answer":"L1 regularization adds the absolute value of weights as a penalty to the loss, encouraging sparsity and pushing some weights to zero, which can perform feature selection.","difficulty":"medium","confidence":0.9}
"""

# ---------------- Prompt templates with few-shot guidance ----------------
MCQ_PROMPT_TEMPLATE = """
You are a strict exam question generator. Use ONLY the CONTEXT blocks to generate EXACTLY {n} multiple-choice questions. Provide distribution: easy={ne}, medium={nm}, hard={nh}.
Return ONLY valid JSON (no explanation) with shape:

{{"questions":[{{"type":"mcq","stem":"...","options":["opt1","opt2","opt3","opt4"],"answer":"optX","difficulty":"easy|medium|hard","confidence":0.0}}, ...]}}

Few-shot example:
{fewshot_example}

Context:
{context}
"""

YESNO_PROMPT_TEMPLATE = """
You are a strict exam question generator. Use ONLY the CONTEXT blocks to generate EXACTLY {n} yes/no questions. Provide distribution: easy={ne}, medium={nm}, hard={nh}.
Return ONLY valid JSON with shape:

{{"questions":[{{"type":"yesno","stem":"...","answer":"Yes" or "No","difficulty":"easy|medium|hard","confidence":0.0}}, ...]}}

Few-shot example:
{fewshot_example}

Context:
{context}
"""

CODE_PROMPT_TEMPLATE = """
You are an exam question generator for coding tasks. Use ONLY the CONTEXT blocks to generate EXACTLY {n} coding questions. Provide distribution: easy={ne}, medium={nm}, hard={nh}.
Return ONLY valid JSON:

{{"questions":[{{"type":"code","stem":"short title","description":"one-paragraph","template":"# starter code","tests":"<PYTEST MODULE TEXT OR JSON CASES>","difficulty":"easy|medium|hard","confidence":0.0}}, ...]}}
Few-shot example:
{fewshot_example}

Context:
{context}
"""

DESCRIPTIVE_PROMPT_TEMPLATE = """
You are an exam question generator. Use ONLY the CONTEXT blocks to generate EXACTLY {n} descriptive questions. Provide distribution: easy={ne}, medium={nm}, hard={nh}.
For each question include 'reference_answer' (2-4 sentence ideal answer).
Return ONLY valid JSON:

{{"questions":[{{"type":"descriptive","stem":"...","reference_answer":"...","difficulty":"easy|medium|hard","confidence":0.0}}, ...]}}
Few-shot example:
{fewshot_example}

Context:
{context}
"""

In [10]:

# ---------------- Difficulty helpers (fixed) ----------------
def difficulty_split_counts(n: int) -> Tuple[int,int,int]:
    """
    Deterministic split: floor-based with remainder distributed to easy -> medium -> hard.
    """
    if n <= 0:
        return 0,0,0
    r_e, r_m, r_h = DIFF_RATIOS
    raw_e = n * r_e
    raw_m = n * r_m
    # raw_h = n * r_h  # we compute via remainder
    e = int(math.floor(raw_e))
    m = int(math.floor(raw_m))
    h = int(math.floor(n - raw_e - raw_m))
    assigned = e + m + h
    remainder = n - assigned
    # distribute remainder to easy, then medium, then hard
    for idx in range(remainder):
        if idx == 0:
            e += 1
        elif idx == 1:
            m += 1
        else:
            h += 1
    return e, m, h

In [11]:


# ---------------- Robust parser: absolute counts OR proportions ----------------
def parse_proportions_or_counts(raw: str, n_total: int) -> Tuple[int,int,int,int]:
    """
    Parse raw input and return integer counts (mcq, yesno, code, desc) summing to n_total.

    Accepts:
     - absolute counts: "2,0,0,1" or "mcq=2 yesno=0 code=0 desc=1"
     - percentages: "40,30,20,10" (converted to proportions)
     - fractions: "0.4 0.3 0.2 0.1"
    """
    s = raw.strip()
    if not s:
        raise ValueError("Empty input")

    # try named key-value pairs
    kv_pattern = re.compile(r"(mcq|yesno|code|desc|descriptive)\s*=\s*([0-9]+(?:\.[0-9]+)?)", flags=re.I)
    kv = {m.group(1).lower(): float(m.group(2)) for m in kv_pattern.finditer(s)}
    if 'descriptive' in kv and 'desc' not in kv:
        kv['desc'] = kv.pop('descriptive')

    if kv and len(kv) >= 4:
        try:
            values = [kv['mcq'], kv['yesno'], kv['code'], kv['desc']]
        except KeyError:
            raise ValueError("Named keys required: mcq, yesno, code, desc (or descriptive)")
        if all(float(v).is_integer() for v in values) and int(sum(values)) == n_total:
            return (int(values[0]), int(values[1]), int(values[2]), int(values[3]))
        # otherwise fallthrough to proportion handling

    # extract bare numbers
    nums = re.findall(r"[0-9]+(?:\.[0-9]+)?", s)
    if len(nums) == 4:
        nums_f = [float(x) for x in nums]
        # absolute counts if they are integers and sum to n_total
        if all(float(x).is_integer() for x in nums_f) and int(sum(nums_f)) == n_total:
            return tuple(int(x) for x in nums_f)
        # treat as percentages or proportions
        ssum = sum(nums_f)
        if ssum > 1.0 and ssum <= 100.0 + 1e-6:
            props = [x / 100.0 for x in nums_f]
        else:
            props = [(x if x <= 1.0 else x / 100.0) for x in nums_f]
        tot = sum(props)
        if tot <= 0.0:
            raise ValueError("Invalid numeric proportions")
        props = [p / tot for p in props]
        raw_counts = [n_total * p for p in props]
        floored = [int(math.floor(x)) for x in raw_counts]
        assigned = sum(floored)
        remainder = n_total - assigned
        fracs = sorted([(raw_counts[i] - floored[i], i) for i in range(4)], reverse=True)
        i = 0
        while remainder > 0:
            idx = fracs[i % 4][1]
            floored[idx] += 1
            remainder -= 1
            i += 1
        return tuple(floored)

    raise ValueError("Could not parse proportions/counts. Provide 4 numbers (mcq,yesno,code,desc).")

In [12]:

# ---------------- RAG generation with difficulty-aware requests (fixed per-batch logic) ----------------
def make_context_text(retrieved_chunks: List[Dict[str, Any]]) -> str:
    parts = []
    for i, ch in enumerate(retrieved_chunks, start=1):
        meta = ch.get("meta", {})
        tag = f"{meta.get('path','')}:chunk_{ch.get('id')}"
        parts.append(f"--- CONTEXT {i} ({tag}) ---\n{ch['text']}\n")
    return "\n".join(parts)

def generate_candidates_with_rag_difficulty(chunks_meta: List[Dict[str, Any]], embed_model: SentenceTransformer,
                                            target: Dict[str,int], top_k: int = TOP_K_DEFAULT) -> List[Dict[str, Any]]:
    candidates = []
    def gen_for_type(prompt_template, qtype, total_needed, fewshot_example):
        nonlocal candidates
        if total_needed <= 0:
            return 0
        created = 0
        for cm in chunks_meta:
            if created >= total_needed:
                break
            kws = cm.get("keywords", [])
            kw_text = " ".join(kws[:6])
            query_text = f"{kw_text} {qtype} question"
            retrieved = retrieve_top_k(query_text, embed_model, top_k=top_k)
            context = make_context_text(retrieved)
            remaining = total_needed - created
            # batch size (small increments for mcq/yesno)
            batch = min(3, remaining) if qtype in ("mcq","yesno") else 1

            # Compute overall remaining difficulty distribution and slice it for this batch
            be_total, bm_total, bh_total = difficulty_split_counts(remaining)
            # allocate into this batch from totals: take as many as possible respecting batch size
            be = min(be_total, batch)
            bm = min(bm_total, max(0, batch - be))
            bh = max(0, batch - be - bm)

            prompt = prompt_template.format(n=batch, ne=be, nm=bm, nh=bh, context=context, fewshot_example=fewshot_example)
            try:
                j = call_openai_json(prompt)
                qs = j.get("questions", [])
                for q in qs:
                    if "difficulty" not in q or q.get("difficulty") not in ("easy","medium","hard"):
                        q["difficulty"] = "medium"
                    q["id"] = str(uuid.uuid4())
                    q["type"] = qtype
                    q["source_chunk"] = cm.get("id")
                    q["meta_context_ids"] = [r.get("id") for r in retrieved]
                    candidates.append(q)
                created += len(qs)
            except Exception as e:
                print(f"[Generation error type={qtype} chunk={cm.get('id')}] {e}")
        return created

    gen_for_type(MCQ_PROMPT_TEMPLATE, "mcq", target.get("mcq",0), FEWSHOT_EXAMPLE_MCQ)
    gen_for_type(YESNO_PROMPT_TEMPLATE, "yesno", target.get("yesno",0), FEWSHOT_EXAMPLE_YESNO)
    gen_for_type(CODE_PROMPT_TEMPLATE, "code", target.get("code",0), FEWSHOT_EXAMPLE_CODE)
    gen_for_type(DESCRIPTIVE_PROMPT_TEMPLATE, "descriptive", target.get("descriptive",0), FEWSHOT_EXAMPLE_DESC)
    return candidates

In [13]:

# ---------------- Deduplication ----------------
def deduplicate_questions(candidates: List[Dict[str, Any]], embed_model: SentenceTransformer, threshold: float = DEDUP_SIM_THRESHOLD) -> List[Dict[str, Any]]:
    if not candidates:
        return []
    stems = [ (c.get("stem","")[:400]) for c in candidates ]
    vecs = embed_model.encode(stems, convert_to_numpy=True, normalize_embeddings=True)
    kept = []
    used = np.zeros(len(candidates), dtype=bool)
    def score(i):
        conf = candidates[i].get("confidence")
        try:
            return float(conf) if conf is not None else 0.0
        except Exception:
            return 0.0
    order = sorted(range(len(candidates)), key=lambda i: -score(i))
    for i in order:
        if used[i]:
            continue
        kept.append(candidates[i])
        vi = vecs[i]
        sims = (vecs @ vi).astype(float)
        similar_idx = np.where(sims >= threshold)[0]
        for j in similar_idx:
            used[j] = True
    return kept

In [14]:


# ---------------- Validate & Assemble with difficulty ordering ----------------
def validate_question(q: Dict[str, Any]) -> Tuple[bool, str]:
    typ = q.get("type")
    if typ == "mcq":
        if not q.get("stem") or not q.get("options") or not q.get("answer"):
            return False, "Missing mcq fields"
        if not isinstance(q.get("options"), list) or len(q.get("options")) != 4:
            return False, "MCQ must have 4 options"
        if q["answer"] not in q["options"]:
            return False, "Answer not in options"
    elif typ == "yesno":
        if q.get("answer") not in ("Yes", "No"):
            return False, "YesNo answer invalid"
    elif typ == "code":
        if not q.get("tests"):
            return False, "Code missing tests"
    elif typ == "descriptive":
        if not q.get("reference_answer"):
            return False, "Descriptive missing reference_answer"
    else:
        return False, f"Unknown type: {typ}"
    return True, "OK"

def assemble_test_from_candidates_with_difficulty(candidates: List[Dict[str, Any]], n_mcq: int, n_yesno: int, n_code: int, n_desc: int) -> Dict[str, Any]:
    bucket = candidates
    needs = {"mcq": n_mcq, "yesno": n_yesno, "code": n_code, "descriptive": n_desc}
    selected = []
    for diff in ("easy","medium","hard"):
        for qtype in ("mcq","yesno","code","descriptive"):
            need = needs[qtype]
            if need <= 0:
                continue
            chosen = [q for q in bucket if q.get("type")==qtype and q.get("difficulty")==diff][:need]
            selected.extend(chosen)
            needs[qtype] -= len(chosen)
    # Validate & answer_key
    valid_questions = []
    answer_key = {}
    for q in selected:
        ok, reason = validate_question(q)
        q["_valid"] = ok
        q["_validation_reason"] = reason
        valid_questions.append(q)
        if q.get("type") in ("mcq","yesno"):
            answer_key[q["id"]] = q.get("answer")
        else:
            answer_key[q["id"]] = None
    # final ordering: sequential difficulty easy->medium->hard
    def diff_order_value(q):
        return {"easy":0,"medium":1,"hard":2}.get(q.get("difficulty","medium"),1)
    valid_questions.sort(key=lambda x: diff_order_value(x))
    test = {
        "test_id": str(uuid.uuid4()),
        "created_at": datetime.utcnow().isoformat()+"Z",
        "question_count": len(valid_questions),
        "questions": valid_questions,
        "answer_key": answer_key
    }
    return test


In [15]:

# ---------------- Grading & presentation ----------------
def grade_descriptive_with_openai(reference_answer: str, student_answer: str, max_tokens: int = 300) -> Dict[str, Any]:
    prompt = f"""
You are an objective grader. Compare the STUDENT_ANSWER below to the REFERENCE_ANSWER.
Return ONLY valid JSON with keys: score (0-100 integer), feedback (short 1-3 sentence suggestion), confidence (0.0-1.0 float).

REFERENCE_ANSWER:
\"\"\"{reference_answer}\"\"\"

STUDENT_ANSWER:
\"\"\"{student_answer}\"\"\"

Scoring rules:
- 90-100: captures nearly all key points.
- 70-89: captures most points; minor omissions.
- 40-69: partial.
- 0-39: incorrect or irrelevant.

Return concise JSON.
"""
    try:
        j = call_openai_json(prompt)
    except Exception:
        return {"score": 0, "feedback": "Auto-grader failed; review manually.", "confidence": 0.0}
    score = j.get("score") if isinstance(j, dict) else None
    feedback = j.get("feedback") if isinstance(j, dict) else None
    confidence = j.get("confidence") if isinstance(j, dict) else None
    try:
        score = int(score) if score is not None else None
    except Exception:
        score = None
    try:
        confidence = float(confidence) if confidence is not None else None
    except Exception:
        confidence = None
    return {"score": score, "feedback": feedback, "confidence": confidence}

def grade_submission_basic(test: Dict[str, Any], submission: Dict[str, Any]) -> Dict[str, Any]:
    total = 0; correct = 0; details = {}
    for q in test["questions"]:
        total += 1
        qid = q["id"]; typ = q["type"]
        correct_ans = test["answer_key"].get(qid)
        student_ans = submission.get(qid)
        ok = False; note = ""; score = None; feedback = None
        if typ in ("mcq","yesno"):
            ok = (student_ans == correct_ans)
            score = 100 if ok else 0
        elif typ == "code":
            ok = False; note = "Code not auto-graded in prototype"; score = None
        elif typ == "descriptive":
            ok = None; score = None
        details[qid] = {"question_type": typ, "correct_answer": correct_ans, "submitted": student_ans, "correct": ok, "score": score, "note": note, "feedback": feedback}
        if ok is True:
            correct += 1
    return {"correct": correct, "total": total, "details": details}

def present_and_collect_answers(test: Dict[str, Any]) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    submission = {}
    desc_grades = {}
    print("\n--- Start answering the generated test ---\n")
    for idx, q in enumerate(test["questions"], start=1):
        qid = q["id"]; qtype = q["type"]; stem = q.get("stem","")
        print(f"\nQuestion {idx} ({qtype}, difficulty={q.get('difficulty')}): {stem}\n")
        if qtype == "mcq":
            options = q.get("options", [])
            labels = ["A","B","C","D"]
            option_map = {}
            for lab,opt in zip(labels,options):
                option_map[lab]=opt
                print(f"  {lab}) {opt}")
            while True:
                ans = input("Enter option (A/B/C/D or full option text): ").strip()
                if not ans:
                    print("Please enter an answer.")
                    continue
                au = ans.upper()
                if au in option_map:
                    submission[qid] = option_map[au]; break
                matched = None
                for opt in options:
                    if ans.strip().lower() == opt.strip().lower():
                        matched = opt; break
                if matched:
                    submission[qid] = matched; break
                print("Input not recognized.")
        elif qtype == "yesno":
            print("  A) Yes\n  B) No")
            while True:
                ans = input("Enter A/B or Yes/No: ").strip().lower()
                if not ans: continue
                if ans in ("a","yes"): submission[qid]="Yes"; break
                if ans in ("b","no"): submission[qid]="No"; break
                print("Unrecognized.")
        elif qtype == "code":
            print("Enter code; finish with a line '<<<END'")
            lines=[]
            while True:
                line=input()
                if line.strip()=="<<<END": break
                lines.append(line)
            submission[qid]="\n".join(lines)
            print("Code saved. Prototype does not execute code.")
        elif qtype == "descriptive":
            print("Enter multi-line descriptive answer; finish with '<<<END'")
            lines=[]
            while True:
                line=input()
                if line.strip()=="<<<END": break
                lines.append(line)
            student_text="\n".join(lines).strip()
            submission[qid]=student_text
            ref = q.get("reference_answer","")
            g = grade_descriptive_with_openai(ref, student_text)
            desc_grades[qid]=g
            print(f"Auto-grade: {g.get('score')}% | Feedback: {g.get('feedback')}")
        else:
            ans=input("Enter answer: ").strip()
            submission[qid]=ans
    print("\n--- Answers collected ---\n")
    return submission, desc_grades

In [16]:

# ---------------- Mixed helpers ----------------
def distribute_counts(n: int) -> Tuple[int,int,int,int]:
    base = n // 4
    rem = n % 4
    mcq = yes = code = desc = base
    order = ['mcq','yes','code','desc']
    for i in range(rem):
        if order[i] == 'mcq': mcq += 1
        elif order[i] == 'yes': yes += 1
        elif order[i] == 'code': code += 1
        elif order[i] == 'desc': desc += 1
    return mcq, yes, code, desc

In [17]:
def evaluate_submission(test: Dict[str, Any], submission: Dict[str, Any],
                        descriptive_pass_threshold: int = 70) -> Dict[str, Any]:
    """
    Evaluate a submission against a generated `test` object.

    - MCQ / YesNo: 1 mark for exact match, 0 otherwise.
    - Code: left as None (not auto-graded in prototype).
    - Descriptive: uses grade_descriptive_with_openai(reference_answer, student_answer)
      to get {'score', 'feedback', 'confidence'}. We add 'pass' (score >= threshold).

    Returns a dict with:
      - per_question: { qid: { type, correct_answer, submitted, mark, score_percent, feedback, pass, note } }
      - aggregates: { objective_marks, objective_max, descriptive_count, descriptive_summary }
    """
    per_q = {}
    objective_marks = 0
    objective_max = 0
    descriptive_summary = []

    for q in test["questions"]:
        qid = q["id"]
        qtype = q["type"]
        correct_ans = test["answer_key"].get(qid)  # may be None for descriptive/code
        student_ans = submission.get(qid)
        entry = {
            "question_type": qtype,
            "correct_answer": correct_ans,
            "submitted": student_ans,
            "mark": None,
            "score_percent": None,
            "feedback": None,
            "confidence": None,
            "pass": None,
            "note": None
        }

        if qtype in ("mcq", "yesno"):
            # objective question: 1 or 0
            entry["mark"] = 1 if (student_ans == correct_ans) else 0
            entry["note"] = "objective"
            objective_marks += entry["mark"]
            objective_max += 1
            entry["feedback"] = "Correct." if entry["mark"] == 1 else f"Incorrect. Correct answer: {correct_ans}"

        elif qtype == "code":
            # prototype: don't execute student code
            entry["mark"] = None
            entry["note"] = "code_not_graded_in_prototype"
            entry["feedback"] = "Code questions are not auto-graded in this prototype."

        elif qtype == "descriptive":
            # call LLM grader (must already exist in your notebook)
            ref = q.get("reference_answer", "")
            # If student didn't answer, send empty string (grader will handle)
            stu_ans = student_ans or ""
            grade_result = grade_descriptive_with_openai(ref, stu_ans)
            # grade_result expected: {"score": int, "feedback": str, "confidence": float}
            score = grade_result.get("score")
            feedback = grade_result.get("feedback")
            conf = grade_result.get("confidence")
            entry["score_percent"] = score
            entry["feedback"] = feedback
            entry["confidence"] = conf
            entry["pass"] = (score is not None and score >= descriptive_pass_threshold)
            entry["mark"] = 1 if entry["pass"] else 0  # optional: give 1 if passes threshold
            entry["note"] = f"descriptive (pass threshold {descriptive_pass_threshold}%)"
            descriptive_summary.append({"qid": qid, "score": score, "pass": entry["pass"], "feedback": feedback})

        else:
            entry["note"] = "unknown_question_type"

        per_q[qid] = entry

    aggregates = {
        "objective_marks": objective_marks,
        "objective_max": objective_max,
        "objective_percent": (objective_marks / objective_max * 100) if objective_max > 0 else None,
        "descriptive_count": len(descriptive_summary),
        "descriptive_summary": descriptive_summary,
        "total_mark_baseline": None,   # optional combined metric (you can define weighting)
    }

    # OPTIONAL: compute a simple combined score: objective marks (count) + descriptive passed count
    # This is only a suggestion — adapt weights as you prefer.
    try:
        descriptive_pass_count = sum(1 for d in descriptive_summary if d.get("pass"))
        aggregates["total_mark_baseline"] = objective_marks + descriptive_pass_count
        aggregates["total_possible_baseline"] = objective_max + len(descriptive_summary)
        aggregates["total_percent_baseline"] = (aggregates["total_mark_baseline"] /
                                               aggregates["total_possible_baseline"] * 100) if aggregates["total_possible_baseline"] > 0 else None
    except Exception:
        pass

    return {"per_question": per_q, "aggregates": aggregates}


In [18]:
def pretty_print_evaluation(eval_result: Dict[str, Any]) -> None:
    """
    Small helper to print the evaluation summary to the console/notebook.
    """
    ag = eval_result["aggregates"]
    print("\n=== EVALUATION SUMMARY ===")
    if ag["objective_max"] is not None:
        print(f"Objective (MCQ/YesNo): {ag['objective_marks']} / {ag['objective_max']}  ({ag['objective_percent']:.1f}% )")
    if ag["descriptive_count"] is not None:
        print(f"Descriptive questions: {ag['descriptive_count']}")
        for d in ag["descriptive_summary"]:
            print(f" - Q {d['qid']}: score={d['score']} pass={d['pass']} | feedback: {d['feedback']}")
    if ag.get("total_mark_baseline") is not None:
        print(f"Baseline total: {ag['total_mark_baseline']} / {ag['total_possible_baseline']}  ({ag['total_percent_baseline']:.1f}%)")
    print("==========================\n")


In [26]:
def generate_detailed_report(test: Dict[str, Any],
                             submission: Dict[str, Any],
                             evaluation: Dict[str, Any] = None,
                             descriptive_pass_threshold: int = 70,
                             save_path: Path | None = None) -> str:
    """
    Create a detailed per-question textual evaluation in the requested format.
    - `test`: the test dict produced by your pipeline (contains `questions` and `answer_key`).
    - `submission`: mapping question_id -> student's answer (strings).
    - `evaluation`: optional result from evaluate_submission(test, submission). If provided,
      the function will use its descriptive feedback/score instead of re-calling the grader.
    - `descriptive_pass_threshold`: used only if evaluation not provided and we grade descriptives here.
    - `save_path`: optional Path to write the plain-text report. If provided, file is written and path returned.

    Returns the generated multi-line report string.
    """
    lines = []
    lines.append("DETAILED PER-QUESTION EVALUATION")
    lines.append("=" * 40)
    per_eval = evaluation.get("per_question") if (evaluation and "per_question" in evaluation) else None

    for idx, q in enumerate(test.get("questions", []), start=1):
        qid = q.get("id")
        qtype = q.get("type")
        stem = q.get("stem", "").strip()
        lines.append(f"\nQuestion {idx}: {stem}")
        lines.append("-" * 40)

        # Student's submitted answer (may be None)
        student_ans = submission.get(qid)
        # Model/correct answer from answer_key (for objective types)
        correct_ans = test.get("answer_key", {}).get(qid)

        if qtype == "mcq":
            # Show model answer (full option text) and the student's reply
            lines.append(f"Model answer: {correct_ans}")
            lines.append(f"Your reply: {student_ans if student_ans is not None else ''}")
            # Decide correctness (prefer evaluation data if present)
            correct_flag = None
            mark = None
            if per_eval and qid in per_eval:
                entry = per_eval[qid]
                mark = entry.get("mark")
                correct_flag = entry.get("mark") == 1
            else:
                # fallback exact-match
                correct_flag = (student_ans == correct_ans)
                mark = 1 if correct_flag else 0
            lines.append(f"Is correct: {'Yes' if correct_flag else 'No'}")
            lines.append(f"Marks: {mark}")
        elif qtype == "yesno":
            lines.append(f"Model answer: {correct_ans}")
            lines.append(f"Your reply: {student_ans if student_ans is not None else ''}")
            correct_flag = None
            mark = None
            if per_eval and qid in per_eval:
                entry = per_eval[qid]
                mark = entry.get("mark")
                correct_flag = entry.get("mark") == 1
            else:
                correct_flag = (str(student_ans).strip().lower() == str(correct_ans).strip().lower())
                mark = 1 if correct_flag else 0
            lines.append(f"Is correct: {'Yes' if correct_flag else 'No'}")
            lines.append(f"Marks: {mark}")
        elif qtype == "descriptive":
            # Model answer (reference) and student's reply
            model_answer = q.get("reference_answer", "")
            lines.append("Answer:")
            if model_answer:
                for line in model_answer.splitlines():
                    lines.append("  " + line)
            else:
                lines.append("  (no reference answer available)")

            lines.append("Your reply:")
            if student_ans:
                for line in str(student_ans).splitlines():
                    lines.append("  " + line)
            else:
                lines.append("  (no answer submitted)")

            # Analysis & score: prefer evaluation info if present
            score = None; feedback = None; conf = None; passed = None
            if per_eval and qid in per_eval:
                entry = per_eval[qid]
                score = entry.get("score_percent")
                feedback = entry.get("feedback")
                conf = entry.get("confidence")
                passed = entry.get("pass")
            else:
                # call grader directly (this uses your grade_descriptive_with_openai function)
                ref = model_answer or ""
                stu = student_ans or ""
                try:
                    g = grade_descriptive_with_openai(ref, stu)
                    score = g.get("score")
                    feedback = g.get("feedback")
                    conf = g.get("confidence")
                    passed = (score is not None and score >= descriptive_pass_threshold)
                except Exception as e:
                    feedback = f"Auto-grader failed: {e}"
            lines.append("Analysis:")
            if feedback:
                # one- or two-line summary
                for fl in str(feedback).splitlines():
                    lines.append("  " + fl.strip())
            else:
                lines.append("  (no feedback)")
            lines.append(f"Score: {score if score is not None else 'N/A'}")
            lines.append(f"Pass: {'Yes' if passed else 'No'}")
        elif qtype == "code":
            lines.append("Note: Code question — will evaluate soon and get back to you.")
            lines.append("Your submission (saved):")
            if student_ans:
                for line in str(student_ans).splitlines():
                    lines.append("  " + line)
            else:
                lines.append("  (no code submitted)")
        else:
            lines.append(f"Unknown question type '{qtype}' - stored submission:")
            lines.append(str(student_ans))

        lines.append("")  # blank line after each question

    report_text = "\n".join(lines)

    # Optionally save to file
    if save_path:
        try:
            with open(save_path, "w", encoding="utf-8") as f:
                f.write(report_text)
        except Exception as e:
            print("Warning: failed to save detailed report:", e)

    return report_text


def print_and_save_detailed_report(test: Dict[str, Any],
                                   submission: Dict[str, Any],
                                   evaluation: Dict[str, Any] = None,
                                   out_dir: Path = OUTPUT_DIR):
    """
    Convenience wrapper: generates report, prints to console, and saves to a timestamped txt file.
    """
    ts = now_ts()
    filename = out_dir / f"detailed_report_{ts}.txt"
    report = generate_detailed_report(test, submission, evaluation=evaluation, save_path=filename)
    print(report)
    print("\nSaved detailed report to:", filename)
    return filename


In [24]:

# ---------------- Main flow (with custom proportions/counts and fixes) ----------------
def main_rag_flow_with_custom_mixed_and_fixes():
    files = FILES
    if not files:
        raise RuntimeError("FILES list is empty. Set FILES variable to your local PDF/DOCX paths.")
    missing = [f for f in files if not Path(f).exists()]
    if missing:
        print("Missing files:", missing)
        raise FileNotFoundError("Missing files.")
    print("Extracting text and creating chunks...")
    chunks_texts=[]
    chunk_meta=[]
    chunk_id = 0
    for f in files:
        print(" -", f)
        txt = extract_text_from_file(f)
        cks = chunk_text_fast(txt, chunk_chars=CHUNK_CHARS, overlap=CHUNK_OVERLAP)
        for i,c in enumerate(cks):
            chunk_meta.append({"id": chunk_id, "text": c, "meta": {"path": f, "page_idx": i}})
            chunks_texts.append(c)
            chunk_id += 1
    if not chunks_texts:
        raise RuntimeError("No text extracted.")
    print("Computing TF-IDF keywords per chunk...")
    keywords_per_chunk = compute_chunk_keywords(chunks_texts, top_k=6)
    for i, kw in enumerate(keywords_per_chunk):
        chunk_meta[i]["keywords"] = kw
    print("Building embeddings + FAISS index (this will take a moment)...")
    embed_model = build_embeddings_index(chunk_meta, model_name=EMBED_MODEL_NAME)
    global CHUNK_META
    CHUNK_META = chunk_meta

    # user prompts
    while True:
        try:
            n = int(input("Enter how many questions to generate (integer): ").strip())
            if n<=0:
                print("Enter a positive integer"); continue
            break
        except Exception:
            print("Invalid input")
    print("Choose question type: a) MCQ  b) Yes/No  c) Code  d) Descriptive  e) Mixed (combination of all)")
    type_map={"a":"mcq","b":"yesno","c":"code","d":"descriptive","e":"mixed"}
    while True:
        choice = input("Enter a/b/c/d/e (or mcq/yesno/code/descriptive/mixed): ").strip().lower()
        if choice in ("a","b","c","d","e"):
            sel = type_map[choice]; break
        if choice in ("mcq","yesno","code","descriptive","mixed"):
            sel = choice; break
        print("Invalid choice.")

    n_mcq = n_yesno = n_code = n_desc = 0
    if sel == "mixed":
        use_custom = input("Use custom proportions or absolute counts for MCQ/YesNo/Code/Descriptive? (y/N): ").strip().lower()
        if use_custom in ("y","yes"):
            raw = input("Enter 4 numbers. Acceptable formats:\n - absolute counts: '2,0,0,1' or 'mcq=2 yesno=0 code=0 desc=1'\n - percentages: '40,30,20,10'\n - fractions: '0.4 0.3 0.2 0.1'\nYour input: ").strip()
            try:
                n_mcq, n_yesno, n_code, n_desc = parse_proportions_or_counts(raw, n)
                print(f"Using counts -> MCQ={n_mcq}, YesNo={n_yesno}, Code={n_code}, Descriptive={n_desc}")
            except Exception as e:
                print("Invalid proportions/counts input:", e)
                print("Falling back to default equal-split distribution.")
                n_mcq, n_yesno, n_code, n_desc = distribute_counts(n)
                print(f"Default counts: MCQ={n_mcq}, YesNo={n_yesno}, Code={n_code}, Descriptive={n_desc}")
        else:
            n_mcq, n_yesno, n_code, n_desc = distribute_counts(n)
            print(f"Default mixed distribution used: MCQ={n_mcq}, YesNo={n_yesno}, Code={n_code}, Descriptive={n_desc}")
    else:
        if sel=="mcq": n_mcq = n
        elif sel=="yesno": n_yesno = n
        elif sel=="code": n_code = n
        elif sel=="descriptive": n_desc = n

    target = {"mcq": n_mcq, "yesno": n_yesno, "code": n_code, "descriptive": n_desc}
    print("Generating candidate questions via RAG (difficulty-aware) ...")
    candidates = generate_candidates_with_rag_difficulty(CHUNK_META, embed_model, target=target, top_k=TOP_K_DEFAULT)
    print("Candidates generated:", len(candidates))
    print("Deduplicating candidates (semantic)...")
    deduped = deduplicate_questions(candidates, embed_model, threshold=DEDUP_SIM_THRESHOLD)
    print("After dedupe:", len(deduped))
    test = assemble_test_from_candidates_with_difficulty(deduped, n_mcq=n_mcq, n_yesno=n_yesno, n_code=n_code, n_desc=n_desc)
    ts = now_ts()
    test_path = OUTPUT_DIR / f"test_{ts}.json"
    save_json(test, test_path)
    save_json(test["answer_key"], OUTPUT_DIR / f"answer_key_{ts}.json")
    print("Saved test JSON:", test_path)
    submission, desc_grades = present_and_collect_answers(test)

    # new evaluation:
    evaluation = evaluate_submission(test, submission, descriptive_pass_threshold=70)
    # generate the report text and print it immediately (also save to file)
    report_text = generate_detailed_report(test, submission, evaluation=evaluation, save_path=OUTPUT_DIR / f"detailed_report_{now_ts()}.txt")
    print(report_text)
    print("Saved detailed report.")


    # generate and print & save the detailed per-question report
    #detailed_file = print_and_save_detailed_report(test, submission, evaluation=evaluation, out_dir=OUTPUT_DIR)

    
    # new evaluation:
    #evaluation = evaluate_submission(test, submission, descriptive_pass_threshold=70)
    #pretty_print_evaluation(evaluation)
    # save the evaluation to disk (example)
    eval_path = OUTPUT_DIR / f"evaluation_{now_ts()}.json"
    save_json(evaluation, eval_path)
    print("Saved evaluation:", eval_path)

    sub_path = OUTPUT_DIR / f"submission_{ts}.json"
    save_json(submission, sub_path)
    basic = grade_submission_basic(test, submission)
    for qid, gi in desc_grades.items():
        if qid in basic["details"]:
            basic["details"][qid]["score"] = gi.get("score")
            basic["details"][qid]["feedback"] = gi.get("feedback")
            basic["details"][qid]["note"] = f"Auto-graded descriptive; confidence {gi.get('confidence')}"
            basic["details"][qid]["correct"] = True if (gi.get("score") or 0) >= 70 else False
    grade_path = OUTPUT_DIR / f"grade_{ts}.json"
    save_json(basic, grade_path)
    print("Saved submission and grade:", sub_path, grade_path)
    print(f"Test id: {test['test_id']}  Questions: {test['question_count']}")
    return {"test": test, "submission": submission, "grade": basic}

In [25]:

# ---------------- Run ----------------
if __name__ == "__main__":
    res = main_rag_flow_with_custom_mixed_and_fixes()

Extracting text and creating chunks...
 - Introduction to Machine Learning with Python ( PDFDrive.com )-min-81-82.pdf
Computing TF-IDF keywords per chunk...
Building embeddings + FAISS index (this will take a moment)...
Embedding chunks (this may take a moment)...


Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  6.61it/s]

FAISS index built with 2 vectors.


Enter how many questions to generate (integer):  3


Choose question type: a) MCQ  b) Yes/No  c) Code  d) Descriptive  e) Mixed (combination of all)


...


Question 2 (yesno, difficulty=easy): Do linear models often perform well when the number of features is large compared to the number of samples?

  A) Yes
  B) No


Enter A/B or Yes/No:  A



Question 3 (descriptive, difficulty=easy): What are the main strengths and weaknesses of linear models in machine learning?

Enter multi-line descriptive answer; finish with '<<<END'


 hi
 <<<END


Auto-grade: 0% | Feedback: The answer is irrelevant and does not address any key points from the reference answer.

--- Answers collected ---

DETAILED PER-QUESTION EVALUATION

Question 1: What is a key advantage of using L1 regularization in linear models?
----------------------------------------
Model answer: It helps in feature selection by using only a few features.
Your reply: It increases model complexity.
Is correct: No
Marks: 0


Question 2: Do linear models often perform well when the number of features is large compared to the number of samples?
----------------------------------------
Model answer: Yes
Your reply: Yes
Is correct: Yes
Marks: 1


Question 3: What are the main strengths and weaknesses of linear models in machine learning?
----------------------------------------
Model answer:
  Linear models are very fast to train and predict, making them suitable for very large datasets and sparse data. They are easy to interpret, especially when using L1 regularization, which